#### Crop DSM to the same size as the Satellite Image

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
import json

def crop_dsm_to_sat_extent(dsm_path, sat_path, output_path):
    """
        Crop the DSM to the same extent as the satellite image.
    """
    
    with rasterio.open(sat_path) as sat_src:
        sat_crs = sat_src.crs
        sat_bounds = sat_src.bounds
        print(f"Satellite image CRS: {sat_crs}")
        print(f"Satellite image bounds: {sat_bounds}")

    with rasterio.open(dsm_path) as dsm_src:
        dsm_crs = dsm_src.crs
        print(f"DSM CRS: {dsm_crs}")

        if sat_crs != dsm_crs:
            print("CRS mismatch, transforming bounds...")

            minx, miny, maxx, maxy = transform_bounds(
                sat_crs, 
                dsm_crs, 
                sat_bounds.left, 
                sat_bounds.bottom, 
                sat_bounds.right, 
                sat_bounds.top
            )
        else:
            minx, miny, maxx, maxy = sat_bounds.left, sat_bounds.bottom, sat_bounds.right, sat_bounds.top

        bbox = box(minx, miny, maxx, maxy)
        geo = [json.loads(json.dumps(bbox.__geo_interface__))]

        print("Cropping the DSM...")
        out_image, out_transform = mask(dsm_src, geo, crop=True)
        
        out_meta = dsm_src.meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1], 
        "width": out_image.shape[2],  
        "transform": out_transform    
    })

    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(out_image)
    
    print(f"finished! The cropped DSM is saved at {output_path}")


dsm_file = "dsm.tif"       # your big size DSM file
satellite_file = "Google19.tif" # your small size satellite image file
output_file = "cropped_dsm.tif"  # ouput cropped DSM file path

try:
    crop_dsm_to_sat_extent(dsm_file, satellite_file, output_file)
except Exception as e:
    print(f"Error occurred: {e}")

#### Get the metadate json file of Thermal-UAV dataset

In [9]:
import os
import csv
import json
from pathlib import Path
import pandas as pd
def convert_img_info_to_json(modality = 'Thermal'):
    """
    transform the Thermal/Visible data into three JSON files (train, test, valid)
    """
    base_path = Path("Data/Thermal-UAV")
    output_path = Path("Data/metadata")
    output_path.mkdir(parents=True, exist_ok=True)
    
    splits = ['train', 'test', 'valid']
    
    for split in splits:
        split_path = base_path / split
        all_data = []

        changsha_path = split_path / "changsha"
        if not changsha_path.exists():
            continue
            
        # traverse each scene directory under changsha
        for scene_dir in sorted(changsha_path.iterdir()):
            if not scene_dir.is_dir():
                continue
                
            img_info_file = scene_dir / f"{modality}_info.csv"
            if not img_info_file.exists():
                print(f"warning: {img_info_file} does not exist, skip")
                continue
            
            print(f"processing: {img_info_file}")
            
            # read the CSV file
            with open(img_info_file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    filename = row['filename']
                    relative_path = f"./Data/Thermal-UAV/{split}/changsha/{scene_dir.name}/{modality}/{filename}"
                    # Thermal image use DJI M4T drone
                    entry = {
                        "name": relative_path,
                        "lat": float(pd.to_numeric(row['lat'])),
                        "lon": float(pd.to_numeric(row['lon'])),
                        'abs_height': float(pd.to_numeric(row['abs_height'])),
                        "rel_height": float(pd.to_numeric(row['rel_height'])),
                        "pitch": float(pd.to_numeric(row['pitch'])),
                        "yaw": float(pd.to_numeric(row['yaw'])),
                        "roll": float(pd.to_numeric(row['roll'])),
                        "cam_size": 9.83,  # physical length of the diagonal
                        "focal_len": 12.0,  
                        "width": 1280.0,  
                        "height": 1024.0,  
                        "fov": 35.4,  
                        "TargetLatitude": float(pd.to_numeric(row['TargetLatitude'])),
                        "TargetLongitude": float(pd.to_numeric(row['TargetLongitude']))
                    }
                    
                    # # Visible image use DJI M4T drone
                    # entry = {
                    #     "name": relative_path,
                    #     "lat": float(pd.to_numeric(row['lat'])),
                    #     "lon": float(pd.to_numeric(row['lon'])),
                    #     'abs_height': float(pd.to_numeric(row['abs_height'])),
                    #     "rel_height": float(pd.to_numeric(row['rel_height'])),
                    #     "pitch": float(pd.to_numeric(row['pitch'])),
                    #     "yaw": float(pd.to_numeric(row['yaw'])),
                    #     "roll": float(pd.to_numeric(row['roll'])),
                    #     "cam_size": 12.11,  # physical length of the diagonal
                    #     "focal_len": 6.73,  
                    #     "width": 4032.0,  
                    #     "height": 3024.0,  
                    #     "fov": 84.0,  
                    #     "TargetLatitude": float(pd.to_numeric(row['TargetLatitude'])),
                    #     "TargetLongitude": float(pd.to_numeric(row['TargetLongitude']))
                    # }
                    
                    all_data.append(entry)
        
        # save json file
        output_file = output_path / f"{split}_{modality}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, indent=4, ensure_ascii=False)
        
        print(f"already generated: {output_file}，containing {len(all_data)} records") # if no this print, it is wrong with the path or the csv file, but no error will be raised, so add this print to check if the file is generated successfully
    
    print("\ncompile done!") 

if __name__ == "__main__":
    # modality = 'Visible'
    modality = 'Thermal'
    convert_img_info_to_json(modality)

processing: Data\Thermal-UAV\train\changsha\city_300_ortho_night\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\city_300_otho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village_300_ortho_day\Thermal_info.csv
processing: Data\Thermal-UAV\train\changsha\village_300_ortho_night\Thermal_info.csv
already generated: Data\metadata\train_Thermal.json，containing 2355 records

compile done!


#### Deal the pkl file to analysis the results

In [10]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

RESULT_ROOT = r'E:\Learning\SCC-Loc\Result\Experiment_20260322_164302\changsha' 
# OUTPUT_DIR = './evaluation_summary'

# standard for evaluation
RETRIEVAL_THRESHOLD = 0.5
TOP_N = 5 # look at the top 10 retrieval results
DIST_THRESHOLDS = [5, 10, 20, 30, 50] # accuracy thresholds for localization recall
PENALTY_VALUE = 1000 # penalty value for failed or missing data

def load_and_parse_pkls(root_dir):
    pkl_files = list(Path(root_dir).rglob("*.pkl"))
    print(f"Found {len(pkl_files)} pkl files in {root_dir}")
    
    data_list = []
    for pkl_path in pkl_files:
        try:
            with open(pkl_path, 'rb') as f:
                data = pickle.load(f)
            
            # retrieval statistics
            pde_list = data.get('PDE', [])
            retrieval_hit = False
            if isinstance(pde_list, (list, np.ndarray)) and len(pde_list) > 0:
                top_n_min_pde = np.min(pde_list[:TOP_N])
                if top_n_min_pde < RETRIEVAL_THRESHOLD:
                    retrieval_hit = True

            # localization error statistics
            err_dict = data.get('detailed_error_dict')
            status = 'Fail'
            h_err, e_alt, e_yaw = PENALTY_VALUE, PENALTY_VALUE, PENALTY_VALUE
            
            if err_dict:
                status = 'Success'
                temp_h_err = err_dict.get('horizontal_error')
                if temp_h_err is None:
                    lat_err = err_dict.get('err_lat')
                    lon_err = err_dict.get('err_lon')
                    if lat_err is not None and lon_err is not None:
                        temp_h_err = np.sqrt(lat_err**2 + lon_err**2)
                
                if temp_h_err is not None and not np.isnan(temp_h_err):
                    h_err = temp_h_err
                
                temp_alt = err_dict.get('err_alt')
                if temp_alt is not None and not np.isnan(temp_alt):
                    e_alt = abs(temp_alt)
                    
                temp_yaw = err_dict.get('err_yaw')
                if temp_yaw is not None and not np.isnan(temp_yaw):
                    e_yaw = abs(temp_yaw)

            # cost time statistics
            time_cost = data.get('total_time', data.get('VG_time_cost', np.nan))

            data_list.append({
                'filename': os.path.basename(data.get('img_path', pkl_path.name)),
                'retrieval_hit': retrieval_hit,
                'horizontal_error': h_err,
                'err_alt': e_alt,
                'err_yaw': e_yaw,
                'status': status,
                'time_cost': time_cost  
            })
        except Exception as e:
            print(f"skipped {pkl_path}: {e}")
        
    return pd.DataFrame(data_list)

def run_report(df):
    if df.empty: return
    total = len(df)
    
    print("\n" + "="*55)
    print(f"📊 Evaluation Summary (Total Samples: {total})")
    print("="*55)

    # --- A. retrieval recall ---
    ret_recall = df['retrieval_hit'].sum() / total * 100
    print(f"1. Retrieval Recall (Top-{TOP_N}, PDE<{RETRIEVAL_THRESHOLD}): {ret_recall:.2f}%")

    # --- B. localization recall ---
    print("\n2. Localization Recall (Full Sample):")
    recall_results = []
    for t in DIST_THRESHOLDS:
        count = ((df['status'] == 'Success') & (df['horizontal_error'] <= t)).sum()
        rate = count / total * 100
        recall_results.append({"Threshold": f"<{t}m", "Success Count": count, "Recall Rate": f"{rate:.2f}%"})
    print(pd.DataFrame(recall_results).to_string(index=False))

    # --- C. error distribution ---
    print(f"\n3. Global Error Distribution (Including Filled with Penalty Value")
    error_stats = df[['horizontal_error', 'err_alt', 'err_yaw']].agg(['mean', 'std', 'median'])
    error_stats.columns = ['horizontal_error(m)', 'err_alt(m)', 'err_yaw(°)']
    print(error_stats.T.to_string())

    # --- [新增] D. cost time statistics ---
    print("\n4. Cost Time Statistics (Time Cost Analysis):")
    # 排除 NaN 值进行计算
    valid_times = df['time_cost'].dropna()
    if not valid_times.empty:
        avg_time = valid_times.mean()
        std_time = valid_times.std()
        med_time = valid_times.median()
        fps = 1.0 / avg_time if avg_time > 0 else 0
        
        print(f"   - mean time (Mean): {avg_time:.4f} seconds/frame")
        print(f"   - standard deviation (Std) : {std_time:.4f} seconds")
        print(f"   - median (Med) : {med_time:.4f} seconds/frame")
        print(f"   - theoretical frame rate (FPS) : {fps:.2f} Hz")
    else:
        print("   - [Warning] did not find valid time_cost data in .pkl files.")
    print("="*55 + "\n")

if __name__ == "__main__":
    df_final = load_and_parse_pkls(RESULT_ROOT)
    run_report(df_final)
    
    # output for ablation study comparison
    # os.makedirs(OUTPUT_DIR, exist_ok=True)
    # df_final.to_excel(os.path.join(OUTPUT_DIR, "detailed_stats.xlsx"), index=False)

Found 2350 pkl files in E:\Learning\SCC-Loc\Result\Experiment_20260322_164302\changsha

📊 Evaluation Summary (Total Samples: 2350)
1. Retrieval Recall (Top-5, PDE<0.5): 96.77%

2. Localization Recall (Full Sample):
Threshold  Success Count Recall Rate
      <5m           1224      52.09%
     <10m           1731      73.66%
     <20m           1888      80.34%
     <30m           1954      83.15%
     <50m           2042      86.89%

3. Global Error Distribution (Including Filled with Penalty Value
                          mean        std     median
horizontal_error(m)  29.643134  79.531132   4.753838
err_alt(m)           23.530080  57.940833  13.521211
err_yaw(°)            5.167174  50.409998   2.092891

4. Cost Time Statistics (Time Cost Analysis):
   - mean time (Mean): 2.2043 seconds/frame
   - standard deviation (Std) : 0.2255 seconds
   - median (Med) : 2.1960 seconds/frame
   - theoretical frame rate (FPS) : 0.45 Hz

